<a href="https://colab.research.google.com/github/ilfpns/PhoSem/blob/main/Phosem_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 폰트 설정 및 라이브러리 추가
import os
import re
import glob
import shutil
import random

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import torchvision.models as tv_models
from torchvision.models import Wide_ResNet50_2_Weights
import torchvision.transforms.functional as TF

from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

!apt-get -qq install fonts-nanum > /dev/null

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

print("폰트 설정 완료:", plt.rcParams['font.family'])

In [ ]:
# @title 파일 불러오기
if not any('Nanum' in f.name for f in fm.fontManager.ttflist):
    os.system("apt-get -qq install fonts-nanum > /dev/null 2>&1")
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

from google.colab import drive
drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/원본2000"
DATA_DIR = "/content/dataset"

marker_path = os.path.join(DATA_DIR, ".src_marker")
need_copy = True
if os.path.exists(DATA_DIR) and os.path.exists(marker_path):
    with open(marker_path) as f:
        prev_src = f.read().strip()
    if prev_src == SRC and len(os.listdir(DATA_DIR)) > 1:
        need_copy = False

if need_copy:
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    os.makedirs(DATA_DIR, exist_ok=True)
    IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp")
    copied, seen_names = 0, set()
    for root, dirs, files in os.walk(SRC, followlinks=False):
        dirs[:] = [d for d in dirs if not os.path.islink(os.path.join(root, d))]
        for f in files:
            if f.lower().endswith(IMG_EXTS) and f not in seen_names:
                shutil.copy2(os.path.join(root, f), os.path.join(DATA_DIR, f))
                seen_names.add(f)
                copied += 1
    with open(marker_path, "w") as f:
        f.write(SRC)
    print(f"복사 완료: {copied}장  (source: {SRC})")
else:
    print(f"이미 준비됨({SRC} 기준): {len(os.listdir(DATA_DIR))}개 항목")

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp")

def parse_filename(path):
    name = os.path.splitext(os.path.basename(path))[0].lower()
    m = re.match(r"^(\d+)[-_]([ox])(.*)$", name)
    if not m:
        return None
    num, ox, rest = m.groups()
    return {"path": path, "num": int(num),
            "label": "good" if ox == "o" else "bad",
            "is_die": "die" in rest}

all_files = sorted(p for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)
                   if p.lower().endswith(IMG_EXTS))
parsed = [info for p in all_files if (info := parse_filename(p))]
unparsed = [p for p in all_files if parse_filename(p) is None]
die_files = [f for f in parsed if f["is_die"]]
good_paths = sorted(f["path"] for f in die_files if f["label"] == "good")
bad_paths = sorted(f["path"] for f in die_files if f["label"] == "bad")

print(f"전체 {len(all_files)}장 | die 사용 {len(die_files)}장 → 양품 {len(good_paths)} / 불량 {len(bad_paths)}")
print(f"제외: die 미포함 {len(parsed)-len(die_files)}장, 규칙 불일치 {len(unparsed)}장")
for p in unparsed[:10]:
    print("  [규칙 불일치]", os.path.basename(p))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

random.seed(42)
shuffled_good = good_paths.copy()
random.shuffle(shuffled_good)
split = int(len(shuffled_good) * 0.8)
bank_good = shuffled_good[:split]
val_good = shuffled_good[split:]
print(f"뱅크용 양품 {len(bank_good)} / 검증용(held-out) {len(val_good)}")

PATCH_INPUT_SIZE = 512
POOL_KERNEL = 1

def load_for_patchcore(path, size=PATCH_INPUT_SIZE):
    img = Image.open(path).convert("RGB")
    orig_size = img.size
    img = img.resize((size, size), Image.BILINEAR)
    return TF.to_tensor(img).unsqueeze(0).to(DEVICE), orig_size

class PatchFeatureExtractor(torch.nn.Module):
    def __init__(self, device=DEVICE, pool_kernel=POOL_KERNEL):
        super().__init__()
        backbone = tv_models.wide_resnet50_2(weights=Wide_ResNet50_2_Weights.IMAGENET1K_V1)
        backbone.eval()
        for p in backbone.parameters():
            p.requires_grad = False
        self.backbone = backbone.to(device)
        self.pool_kernel = pool_kernel
        self._feat = {}
        self.backbone.layer2.register_forward_hook(self._hook('layer2'))
        self.backbone.layer3.register_forward_hook(self._hook('layer3'))

    def _hook(self, name):
        def fn(module, inp, out):
            self._feat[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x):
        self._feat = {}
        _ = self.backbone(x)
        f2, f3 = self._feat['layer2'], self._feat['layer3']
        f3_up = F.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
        combined = torch.cat([f2, f3_up], dim=1)
        pad = self.pool_kernel // 2
        return F.avg_pool2d(combined, kernel_size=self.pool_kernel, stride=1, padding=pad)

extractor = PatchFeatureExtractor(pool_kernel=POOL_KERNEL)
print(f"extractor 준비 (PATCH_INPUT_SIZE={PATCH_INPUT_SIZE}, POOL_KERNEL={POOL_KERNEL})")

@torch.no_grad()
def patchcore_score(path, memory_bank, extractor, size=PATCH_INPUT_SIZE, chunk=2048):
    x, orig_size = load_for_patchcore(path, size)
    feat = extractor(x)
    B, C, H, W = feat.shape
    patches = feat.permute(0, 2, 3, 1).reshape(H*W, C)
    dists = []
    for i in range(0, patches.shape[0], chunk):
        d = torch.cdist(patches[i:i+chunk], memory_bank)
        dists.append(d.min(dim=1).values)
    dists = torch.cat(dists)
    heatmap = F.interpolate(dists.reshape(1,1,H,W), size=(size,size),
                            mode='bilinear', align_corners=False)
    return heatmap.squeeze().cpu().numpy(), dists.max().item(), orig_size

def sweep_threshold(df, n=50):
    ths = np.linspace(df.이상점수.min(), df.이상점수.max(), n)
    rows = []
    for th in ths:
        pred_bad = df.이상점수 > th
        actual_bad = df.실제 == '불량'
        tp = (pred_bad & actual_bad).sum(); fn = (~pred_bad & actual_bad).sum()
        fp = (pred_bad & ~actual_bad).sum(); tn = (~pred_bad & ~actual_bad).sum()
        rows.append({'threshold': round(th,2), 'accuracy': round((tp+tn)/len(df),3),
                     '불량_recall': round(tp/(tp+fn),3) if tp+fn else 0,
                     '놓친_불량': fn, '과검_양품': fp})
    return pd.DataFrame(rows)

print("초기화 완료")

import os, glob, shutil

SRC = "/content/drive/MyDrive/원본2000"
DATA_DIR = "/content/dataset"

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR)

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp")

copied, seen_names = 0, set()
for root, dirs, files in os.walk(SRC, followlinks=False):
    dirs[:] = [d for d in dirs if not os.path.islink(os.path.join(root, d))]
    for f in files:
        if f.lower().endswith(IMG_EXTS) and f not in seen_names:
            shutil.copy2(os.path.join(root, f), os.path.join(DATA_DIR, f))
            seen_names.add(f)
            copied += 1

print(f"복사 완료: {copied}장 (중복 이름 제외)")

In [ ]:
# @title Github repo 확인

repo_path = "/content/Phosem-CNN-Problem-Detection"
if not os.path.exists(repo_path):
    !git clone https://github.com/Phosem/Phosem-CNN-Problem-Detection.git {repo_path}
    print(f"Repository cloned to {repo_path}")
else:
    print(f"Repository already exists at {repo_path}")

if os.path.exists(os.path.join(repo_path, 'data_loader.py')):
    print("data_loader.py found.")
else:
    print("Error: data_loader.py not found in the cloned repository.")

In [ ]:
# @title 이미지 종류 분리
IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp")

def parse_filename(path):
    """
    '429-x_die800.png' → {'num': 429, 'label': 'bad', 'is_die': True}
    '431-o.png'        → {'num': 431, 'label': 'good', 'is_die': False}
    규칙에 안 맞으면 None
    """
    name = os.path.splitext(os.path.basename(path))[0].lower()
    m = re.match(r"^(\d+)[-_]([ox])(.*)$", name)
    if not m:
        return None
    num, ox, rest = m.groups()
    return {
        "path": path,
        "num": int(num),
        "label": "good" if ox == "o" else "bad",
        "is_die": "die" in rest,
    }

all_files = sorted(
    p for p in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True)
    if p.lower().endswith(IMG_EXTS)
)

parsed    = [info for p in all_files if (info := parse_filename(p))]
unparsed  = [p for p in all_files if parse_filename(p) is None]

die_files  = [f for f in parsed if f["is_die"]]
good_paths = sorted(f["path"] for f in die_files if f["label"] == "good")
bad_paths  = sorted(f["path"] for f in die_files if f["label"] == "bad")

print(f"전체 이미지        : {len(all_files)}장")
print(f"  ├ die 포함(사용) : {len(die_files)}장  →  양품 {len(good_paths)} / 불량 {len(bad_paths)}")
print(f"  ├ die 미포함(제외): {len(parsed) - len(die_files)}장")
print(f"  └ 규칙 불일치     : {len(unparsed)}장")

if unparsed:
    print("\n[확인 필요] 파일명 규칙에 안 맞는 파일:")
    for p in unparsed[:10]:
        print("   ", os.path.basename(p))

assert good_paths, "양품(die) 이미지가 없습니다 — PatchCore 메모리 뱅크에 필수"
print("\n양품 예시:", [os.path.basename(p) for p in good_paths[:3]])
print("불량 예시:", [os.path.basename(p) for p in bad_paths[:3]])

In [ ]:
# @title 이미지 학습 / 검증 데이터 분리
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

IMG_SIZE = 256
class DieDataset(Dataset):
    def __init__(self, paths, labels, train=True):
        self.paths, self.labels = paths, labels
        if train:
            self.tf = T.Compose([
                T.Resize((IMG_SIZE, IMG_SIZE)),
                T.RandomHorizontalFlip(),
                T.RandomVerticalFlip(),
                T.RandomRotation(10),
                T.ColorJitter(brightness=0.2, contrast=0.2),
                T.ToTensor(),
            ])
        else:
            self.tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])

    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.labels[i]

all_paths  = good_paths + bad_paths
all_labels = [0]*len(good_paths) + [1]*len(bad_paths)

idx = list(range(len(all_paths)))
random.seed(42); random.shuffle(idx)
split = int(len(idx)*0.8)
tr_idx, va_idx = idx[:split], idx[split:]

train_ds = DieDataset([all_paths[i] for i in tr_idx], [all_labels[i] for i in tr_idx], train=True)
val_ds   = DieDataset([all_paths[i] for i in va_idx], [all_labels[i] for i in va_idx], train=False)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=8)

print(f"학습 {len(train_ds)}장 / 검증 {len(val_ds)}장")
print(f"검증 세트 불량 비율: {sum(all_labels[i] for i in va_idx)}/{len(va_idx)}")

In [ ]:
# @title 자체 build_cnn 모델 구성
import torch.nn as nn

def build_cnn(channels=[16, 32, 64], num_classes=2, in_size=IMG_SIZE):
    layers, in_ch = [], 3
    for out_ch in channels:
        layers += [
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        ]
        in_ch = out_ch
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, num_classes)]
    return nn.Sequential(*layers)

model = build_cnn([8, 16, 32]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"파라미터 수: {n_params:,}")

"""
Layers = 3
dimension = 2
Channel = [8, 16, 32]
"""

SyntaxError: incomplete input (3657487337.py, line 3)

In [ ]:
# @title 학습
import torch.nn as nn
from torch.utils.data import DataLoader


@torch.no_grad()
def evaluate(model, val_dl, device):
    """불량=1 기준 recall/precision/accuracy 계산"""
    model.eval()
    tp = fp = fn = correct = total = 0
    for x, y in val_dl:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += len(x)
        tp += ((pred == 1) & (y == 1)).sum().item()
        fp += ((pred == 1) & (y == 0)).sum().item()
        fn += ((pred == 0) & (y == 1)).sum().item()
    recall = tp / (tp + fn) if tp + fn else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    return correct / total, recall, precision


def train_model(model, train_ds, val_dl, device,
                epochs=30, lr=1e-3, batch_size=32,
                patience=7, save_path="best_classifier.pth"):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    best = {"recall": -1, "precision": -1, "acc": -1, "epoch": -1}
    no_improve, history = 0, []

    for ep in range(1, epochs + 1):
        train_ds.set_epoch(ep - 1)
        train_dl = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=2)

        model.train()
        tr_loss = 0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(x)
        tr_loss /= len(train_ds)

        acc, recall, precision = evaluate(model, val_dl, device)
        history.append((tr_loss, acc, recall, precision))

        improved = (recall, precision) > (best["recall"], best["precision"])
        if improved:
            best.update(recall=recall, precision=precision, acc=acc, epoch=ep)
            torch.save(model.state_dict(), save_path)
            no_improve = 0
        else:
            no_improve += 1

        print(f"epoch {ep:2d} | loss {tr_loss:.4f} | acc {acc:.3f} "
              f"| recall {recall:.3f} | precision {precision:.3f}"
              + (" *" if improved else ""))

        if no_improve >= patience:
            print(f"-> {patience} epoch 개선 없음, 조기 종료")
            break

    print(f"\nbest (epoch {best['epoch']}): recall {best['recall']:.3f}, "
          f"precision {best['precision']:.3f}, acc {best['acc']:.3f} "
          f"-> {save_path} 저장됨")
    return history, best